[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanenalmayouf/applied-ml-workshop/blob/main/labs_colab/day3/lab_3_manafeth_models.ipynb)

# 🚦 مختبر اليوم الثالث — تصنيف مغادرة العملاء وانحدار سعر المركبة

**ورشة أسس تعلم الآلة التطبيقي — اليوم 3 من 5**

هذا الدفتر مبني على مواصفات مختبرات «منافذ» المعتمدة للدورة، ومُجهَّز للعمل مباشرة في **Google Colab** أو في Jupyter محليًا.

**كيف تفتحه في Colab:** من قائمة `File → Upload notebook` في Colab ارفع هذا الملف. إذا لم يجد الدفتر مجلد البيانات تلقائيًا، ستظهر لك خانة لرفع ملف حزمة البيانات (`manafeth_data_package.zip`) المرفق بجوار هذا الدفتر — ارفعه وسيُستكمل التحميل تلقائيًا.

> 📌 راجع `00_start_here.md` قبل البدء لمعرفة طريقة استخدام خلايا **فكّر أولًا** و**TODO** و**مساعدة** في هذه الدفاتر.

In [ ]:
from pathlib import Path
import pandas as pd

# 1) نبحث عن مجلد البيانات بجانب هذا الدفتر (يعمل محليًا وفي Colab إذا رفعت المجلد كاملًا)
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data"), Path("../data/raw"), Path("data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)

# 2) إذا لم نجد المجلد ونحن داخل Google Colab، نطلب من الطالب رفع حزمة البيانات (ملف zip)
if DATA_DIR is None:
    try:
        from google.colab import files
        import zipfile

        print("لم يتم العثور على مجلد البيانات محليًا.")
        print("ارفع ملف حزمة البيانات (.zip) الذي يرافق هذا المختبر ثم انتظر انتهاء الرفع...")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(name) as z:
                    z.extractall(".")
        DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), Path("manafeth_data_package"))
    except ImportError:
        DATA_DIR = Path("manafeth_data_package")
        print("تنبيه: لسنا داخل Google Colab ولم يوجد مجلد بيانات — ضع حزمة البيانات بجانب الدفتر.")

CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"

print("مجلد البيانات المستخدم:", DATA_DIR.resolve())
assert CUSTOMERS_PATH.exists(), "تعذّر العثور على manafeth_customers.parquet — تأكد من رفع حزمة البيانات كاملة."

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (7, 4)

## 🎯 هدف المختبر

تنفّذ اليوم مهمتين على بيانات مختلفة من الحزمة نفسها: **تصنيف** مغادرة عميل، و**انحدار** سعر مركبة مستعملة. الهدف هو أن تثبّت الفرق بين توقّع **فئة** وتوقّع **رقم ذي مقدار** — لا الوصول إلى أفضل أداء ممكن.

## السيناريو والبيانات

| المسار | السؤال | الملف | الهدف | نوع المهمة |
|---|---|---|---|---|
| التصنيف | هل سيغادر العميل خلال 30 يومًا؟ | `manafeth_customers.parquet` | `churned_30d` | تصنيف ثنائي |
| الانحدار | ما سعر البيع التقريبي للمركبة؟ | `markabat_listings_sample.csv` (2,000 إعلان) | `sale_price_sar` | انحدار |

## 🤔 فكّر أولًا

ملف المركبات يحتوي على `listed_month` و`days_on_platform`. لماذا نستبعدهما من أول نموذج للسعر مع أن بقاء الإعلان مدة أطول قد يرتبط فعليًا بسعره؟ (فكّر: هل تعرف هاتين القيمتين لحظة إنشاء الإعلان؟)

### الجزء الأول — تصنيف مغادرة العملاء

أعد استخدام `preprocessor` وتقسيم التدريب/الاختبار من اليوم الثاني، وضعه داخل خط أنابيب مع انحدار لوجستي.

In [ ]:
customers = pd.read_parquet(CUSTOMERS_PATH)

safe_features = [
    "city", "city_tier", "device", "payment_method",
    "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating", "last_promo_used"
]
numeric_features = [
    "city_tier", "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating"
]
categorical_features = ["city", "device", "payment_method", "last_promo_used"]

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

X = customers[safe_features]
y = customers["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ("numbers", Pipeline([
        ("fill", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler())
    ]), numeric_features),
    ("categories", Pipeline([
        ("fill", SimpleImputer(strategy="constant", fill_value="غير_معروف")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression

# TODO: ابنِ Pipeline باسمي المرحلتين "prepare" (=preprocessor) و"model" (=LogisticRegression(max_iter=1000))
churn_classifier = Pipeline([
    ("prepare", ...),
    ("model", ...)
])
churn_classifier.fit(X_train, y_train)

# TODO: احسب الفئات المتوقعة بـ predict() والاحتمالات بـ predict_proba()[:, 1]
churn_predictions = ...
churn_probabilities = ...
print("أول 5 فئات متوقعة:", churn_predictions[:5])
print("أول 5 احتمالات مغادرة:", churn_probabilities[:5].round(3))

### 📊 رسم: توزيع احتمالات المغادرة المتوقعة

مثال مُنفَّذ — يوضح كيف يوزّع النموذج احتمالات المغادرة على عملاء الاختبار.

In [ ]:
plt.hist(churn_probabilities, bins=30, color="#5B4FCF")
plt.title("توزيع احتمالات المغادرة المتوقعة على عملاء الاختبار")
plt.xlabel("احتمال المغادرة")
plt.ylabel("عدد العملاء")
plt.show()

### الجزء الثاني — انحدار سعر المركبة

ملف مختلف تمامًا، وهدف مختلف تمامًا: رقم مستمر بدل فئة.

In [ ]:
vehicles = pd.read_csv(VEHICLES_PATH)
vehicles.head()

In [ ]:
vehicle_target = "sale_price_sar"

# TODO: استبعد listing_id (معرّف)، وlisted_month وdays_on_platform (غير معروفين لحظة إنشاء الإعلان) من الخصائص
vehicle_features = [
    # اكتب أسماء الأعمدة المتبقية والمناسبة كخصائص
]

X_vehicle = vehicles[vehicle_features]
y_vehicle = vehicles[vehicle_target]
Xv_train, Xv_test, yv_train, yv_test = train_test_split(
    X_vehicle, y_vehicle, test_size=0.20, random_state=42
)

# TODO: قسّم أعمدة المركبات إلى رقمية وفئوية وابنِ vehicle_preprocessor بنفس أسلوب اليوم الثاني
vehicle_numeric = [...]
vehicle_categorical = [...]
vehicle_preprocessor = ColumnTransformer([
    ("numbers", Pipeline([("fill", ...), ("scale", ...)]), vehicle_numeric),
    ("categories", Pipeline([("fill", ...), ("encode", ...)]), vehicle_categorical)
])

In [ ]:
price_regressor = Pipeline([
    ("prepare", vehicle_preprocessor),
    ("model", LinearRegression())
])
price_regressor.fit(Xv_train, yv_train)
price_predictions = price_regressor.predict(Xv_test)
print("أول 5 أسعار متوقعة (ريال):", price_predictions[:5].round(2))

### 📊 رسم: السعر الفعلي مقابل السعر المتوقع

كل نقطة مركبة واحدة من الاختبار. كلما اقتربت النقاط من الخط المتقطع (y=x) كان التوقع أدق.

In [ ]:
# TODO: ارسم مخطط تشتت (scatter) بين yv_test (المحور الأفقي) وprice_predictions (المحور الرأسي)
# لمساعدتك: أضف خط متقطع y=x بنفس حدود القيم الدنيا/العليا للمقارنة البصرية

## النتيجة المتوقعة

قائمة فئات واحتمالات لمغادرة العملاء، وقائمة أسعار رقمية متوقعة للمركبات. **لا تُعلن أن نموذجًا جيدًا من قراءة خمسة صفوف أو نظرة على الرسم** — قياس الجودة الفعلي محور اليوم الرابع.

## المهارات التي راجعتها

اختيار التصنيف أو الانحدار من شكل الهدف، وضع نموذج داخل `Pipeline`، استخدام `fit`/`predict`/`predict_proba`، والتأكد من أن المعرّفات وأعمدة المستقبل لا تدخل الخصائص.

## ✅ تحقق ذاتيًا قبل إغلاق الدفتر
- [ ] `churn_probabilities` كلها بين 0 و1
- [ ] `vehicle_features` لا يحتوي على `listing_id` ولا `listed_month` ولا `days_on_platform`
- [ ] تستطيع أن تكتب جملتين: «مخرج التصنيف …» و«مخرج الانحدار …» تُبيّنان الفرق بين فئة واحتمال ورقم

**غدًا:** نقيس فعليًا جودة نموذج المغادرة، ونقارن بين عدة نماذج بمقياس مناسب لعدم توازن الفئات.